[Home](../../README.md)

### Feature Engineering

This Jupyter Notepad is a selection of data engineering processes you can apply to your data before model training to maximise the performance of your machine learning model. For this demonstration we will engineer new or improved features for the diabetes data you previously wrangled.

#### Feature Engineering Process
- Deriving new variables from existing ones
    - Encoding categorical features
    - Calculating new features from existing features
- Combining features/feature interactions
- Identifying the most relevant features for the model
- Transforming Features
  - [Dividing Data into categories](https://web.ma.utexas.edu/users/mks/statmistakes/dividingcontinuousintocategories.html)
  - Mathematical transformations (for example logarithmic transformations). Logarithmic transformations are a powerful tool in the world of statistical analysis. They are often used to transform data that exhibit skewness or other irregularities, making it easier to analyze, visualize, and interpret the results.
- Creating Domain-Specific Features that incorporating knowledge from the specific domain to create features that capture important characteristics of the data.

#### Load the required dependencies

In [37]:
# Import frameworks
import pandas as pd
from datetime import date

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [38]:
results = pd.read_csv("2.2.1.wrangled_results.csv")
results_all_events = pd.read_csv("2.2.1.unwrangled_results.csv")
persons = pd.read_csv("2.2.1.wrangled_persons.csv")
countries = pd.read_csv("2.2.1.wrangled_countries.csv")
competitions = pd.read_csv("2.2.1.wrangled_competitions.csv")
current_date = date.today().year

#### Calculating Competitions entered

Competitions entered represents the total number of official WCA competitions a participant has attended across their entire cubing career. Unlike features such as years_active, which measures a form of time, this feature will capture the engangement intensity: how frequently a competitor participates in competitions

This feature is a very importannt behavioural indicator. Some cubers attend only a handful of competitions despite being active for many years, while others compete dozens of times per year. The number of competitions entered reflects not only the dedication to the craft, but also the exposure to competition pressure, judging environmets, and the iterative improvement that comes from repeated official WCA attemptes.

**Why this is useful**:
* Practice in pressure conditions: Frequent competition gives cubers more experience performing solves in regulated high-stakes WCA conditions
* Consitency: More competitions often correlate wiht more stable performance and reduced variamce in solve times
* Interaction with other features: When combined wiht years active, it helps distinguish between long-term but inactive cubers and highly engaging competitors. 
* Access to competitions: This feature can also potentially reflect geographical accessibility, as some regions host far more competitions than others. 

In [39]:
competitions_entered = results_all_events.groupby("person_id")["competition_id"].nunique().reset_index()
competitions_entered.columns = ["person_id", "competitions_entered"]
Q1 = competitions_entered["competitions_entered"].quantile(0.25)
Q3 = competitions_entered["competitions_entered"].quantile(0.95)
IQR = Q3 - Q1
competitions_entered = competitions_entered[(competitions_entered['competitions_entered'] >= Q1 - 1.5 * IQR) & (competitions_entered['competitions_entered'] <= Q3 + 1.5 * IQR)]
print(competitions_entered["competitions_entered"].describe())
results.head()

count    277022.000000
mean          2.951397
std           4.017212
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max          31.000000
Name: competitions_entered, dtype: float64


,pos,best,average,competition_id,event_id,person_name,person_id,person_country_id
0,15,1968,0.081868,LyonOpen2007,333,Etienne Amany,2007AMAN01,Cote d_Ivoire
1,16,1731,0.082378,LyonOpen2007,333,Thomas Rouault,2004ROUA01,France
2,17,2305,0.103482,LyonOpen2007,333,Antoine Simon-Chautemps,2005SIMO01,France
3,19,2677,0.114904,LyonOpen2007,333,Marlène Desmaisons,2007DESM01,France
4,20,1869,0.115074,LyonOpen2007,333,Ton Dennenbroek,2003DENN01,Netherlands


#### Calculating Years Active

Years active represents the total duration of a competitior's involvement in the official WCA competitions. This feature will capture the experience horizon of each participant, turning how long they have been apart of the speedcommunity into a numerical value. 

This matters because longevity often correlates with a deeper understanding of algorithms, improvements in finger dexterity, and the exposure to a wider range of opponents and solving strategies. Even if a competitor participates infrequentlc, simply being active over many years anc contribute to the refinement of their skills. 

**Why this is useful**:
* Experience: Longer active periods often indicate more time spent practicing and learning
* Skill maturation: Cubers typically improve over time, even outside competitions
* Context for other features: Helps normalise counts like competitions entered or events completed over their active lifetime

In the context of experience and comparing averages of each participant, a cubers years active in the competitions has boundless influence on the target. Thus, we will convert their first competition entered into a year, and minus that from the current year by importing datetime.

In [40]:
results["competition_id"] = results["competition_id"].astype(str)
competitions["id"] = competitions["id"].astype(str)

year_results = results.merge(competitions[["id", "year"]], left_on="competition_id", right_on="id", how="left")
years_active = (year_results.groupby("person_id")["year"].agg(["min", "max"]).reset_index())

years_active["years_active"] = current_date - years_active["min"]
years_active = years_active[["person_id", "years_active"]]
print(years_active.head())

    person_id  years_active
0  1982PETR01            19
1  1982RAZO01            19
2  2003BABC01            19
3  2003BADI01            19
4  2003BRUC01            19


#### Person Country

A competitor's country provides important contextual information about their training environment, access to competitions, and regional cubing cultiure. Speedcubing communities vary significantly across different parts of the world - some having much denser competition calendars and strong coaching networks, while others have limited events and smaller communities. 

Extracting the persons country from the WCA dataset allows us to encorporate these geographic influences into the model. 

**Why this is useful**:
* Competition availability: Some countries host dozens of competitions pey year while others only host a few
* Community strength: Stronger local communities often accelerate learning
* Regional performance trends: Certain countries consistently produce top performers due to established training systems

In [41]:
person_country = year_results.merge(competitions[["id", "country_id"]], left_on="competition_id", right_on="id", how="left")
person_country = person_country.groupby("person_id")["country_id"].agg(lambda avg: avg.mode()[0]).reset_index()
person_country.columns = ["person_id", "country"]
print(person_country.head())

    person_id      country
0  1982PETR01      Hungary
1  1982RAZO01      Hungary
2  2003BABC01          USA
3  2003BADI01       France
4  2003BRUC01  Netherlands


#### Person Continent

While country-level data is informative, continent-level grouping captures proader regional patterns. Continents differ in competition density, travek accessibilitym and community size. For example, Europe and Asia have extremely avtive dense cubing circuits, while Africa and South America have fewer events overall. 

By mapping each compeititor's country to its continent, we can create a higher-level categorical feature that smooths out country-specific noise while preserving the meaningful structure of the compeititors demographic. 

**Why this is useful**:
* Larger level trends: Helps identify large-scale performance differences across regions.
* Accessibility: Competitors in densly populated continents may attend more competitions. 
* Generalisation: Continents can become very useful when country categoriess are too granular or sparce

In [42]:
person_continent = person_country.merge(countries[["id", "continent_id"]], left_on="country", right_on="id", how="left")
person_continent = person_continent[["person_id", "continent_id"]]
person_continent.columns = ["person_id", "continent_id"]
print(person_continent.head())

    person_id    continent_id
0  1982PETR01         _Europe
1  1982RAZO01         _Europe
2  2003BABC01  _North America
3  2003BADI01         _Europe
4  2003BRUC01         _Europe


## Combining features/feature interactions

While individual features can be powerful predictors, their interactions often carry even more information. Feature interaction engineering is the process of creating new features that represent the interaction between two or more features.

In this case, dividing the total competitions entered within their respected cubing careers (every competition they have entered since their first competition to their last) over their years active in the community (total years since their first competition), derives for us their average competitions per year.

#### Calculating Competitions Per Year


In [43]:
competitions_per_year = competitions_entered.merge(years_active, on="person_id", how="left")
competitions_per_year["competitions_per_year"] = round(competitions_per_year["competitions_entered"] / competitions_per_year["years_active"].replace(0,1), 2)
competitions_per_year = competitions_per_year.dropna(subset=["years_active", "competitions_per_year"])
print(competitions_per_year.head())

     person_id  competitions_entered  years_active  competitions_per_year
23  2003BABC01                     6          19.0                   0.32
24  2003BADI01                    23          19.0                   1.21
39  2003EAST01                     5          18.0                   0.28
40  2003FALM01                     3          19.0                   0.16
44  2003GRAN01                     2          19.0                   0.11


## Domain specific features

Domain knowledge is about understanding the domain or subject area of the data. In This case the domain is speedcubing and more specifically, in this case, speedcubing algorithms, and the similarities between them. 

Calculating the number of events entered counts more than just the perticipants 3x3x3 entries, as they cold have participated in events such as 2x2x2, 4x4x4, 5x5x5, pyraminx, skewb, and many more. However it is known that many solving algorithms are similar if not exactly the same, no matter the cube. Hence, having experience in other solving categories, before or after having a 3x3x3 attempt, significantly increases the perticipants experience and would therefore decrease their solve time.  

#### Calculating Events Completed

In [44]:
events_completed = results_all_events.groupby("person_id")["event_id"].count().reset_index()
events_completed = events_completed[events_completed["person_id"].isin(results["person_id"])]
events_completed.columns = ["person_id", "events_completed"]
print(events_completed.head())

     person_id  events_completed
8   1982PETR01                91
9   1982RAZO01                95
25  2003BABC01                32
26  2003BADI01                90
37  2003BRUC01              2392


#### Calculating Events Per Competition  

In [45]:
events_per_competition = competitions_entered.merge(events_completed, on="person_id", how="left")
events_per_competition["events_per_competition"] = round(events_per_competition["events_completed"] / events_per_competition["competitions_entered"].replace(0,1), 2)
events_per_competition = events_per_competition.dropna(subset=["events_completed", "events_per_competition"])
print(events_per_competition.head())

     person_id  competitions_entered  events_completed  events_per_competition
23  2003BABC01                     6              32.0                    5.33
24  2003BADI01                    23              90.0                    3.91
39  2003EAST01                     5              10.0                    2.00
40  2003FALM01                     3               3.0                    1.00
44  2003GRAN01                     2               2.0                    1.00


#### MERGE


In [46]:
data_frame = results.copy()

data_frame = data_frame.merge(competitions_entered, on="person_id", how="left")
data_frame = data_frame.merge(years_active, on="person_id", how="left")
data_frame = data_frame.merge(events_completed, on="person_id", how="left")
data_frame = data_frame.merge(competitions_per_year[["person_id", "competitions_per_year"]], on="person_id", how="left")
data_frame = data_frame.merge(person_country, on="person_id", how="left")
data_frame = data_frame.merge(person_continent, on="person_id", how="left")
data_frame = data_frame.merge(events_per_competition[["person_id", "events_per_competition"]], on="person_id", how="left")
data_frame = data_frame.drop(columns=["pos", "best", "competition_id", "event_id", "person_name", "person_country_id"])

data_frame = data_frame.dropna(subset=["competitions_entered", "years_active", "events_completed", "competitions_per_year", "events_per_competition"])

print(data_frame.head())

    average   person_id  competitions_entered  years_active  events_completed  \
0  0.081868  2007AMAN01                   2.0            19                 8   
1  0.082378  2004ROUA01                   6.0            19                14   
2  0.103482  2005SIMO01                  28.0            19               175   
3  0.114904  2007DESM01                  10.0            19                29   
5  0.123609  2007CORN01                   2.0            19                 3   

   competitions_per_year country continent_id  events_per_competition  
0                   0.11  France      _Europe                    4.00  
1                   0.32  France      _Europe                    2.33  
2                   1.47  France      _Europe                    6.25  
3                   0.53  France      _Europe                    2.90  
5                   0.11  France      _Europe                    1.50  


#### Save the wrangled and engineered data to CSV

In [47]:
data_frame.to_csv('../2.3.Model_Training/2.3.1.model_ready_data.csv', index=False)